# Digital Twin de un motor eléctrico

Notebook ejecutable en Google Colab con datos simulados. El ejemplo combina un modelo físico térmico, una estimación del estado del equipo y una representación visual interactiva del gemelo digital.

**Objetivo:** demostrar cómo un motor real o simulado puede tener un equivalente virtual que refleje su carga, temperatura, salud y riesgo de anomalía.

## tl;dr

El notebook simula 24 horas de operación de un motor. Durante la última parte se introduce degradación: aumenta la fricción y disminuye la eficiencia. El gemelo digital calcula una temperatura esperada mediante un balance térmico y compara esa expectativa con las mediciones simuladas.

Al ejecutar todas las celdas se obtienen:
- Datos sintéticos de sensores.
- Estimación de salud y detección de anomalías.
- Gráficas de temperatura, vibración, carga y salud.
- Una representación visual interactiva del motor con indicadores de estado.

## Contexto y método

### ¿Qué es un digital twin?

Un **digital twin** o gemelo digital es una representación virtual de un objeto, máquina o proceso real. No es solamente un dibujo en 3D: también contiene datos, reglas y cálculos que intentan describir lo que sucede en el equipo real.

En este ejemplo imaginamos un motor eléctrico instalado en una planta. El motor físico tiene sensores de temperatura, corriente, velocidad y vibración. El gemelo digital recibe esos datos, calcula lo que debería estar ocurriendo y avisa cuando la realidad comienza a separarse de lo esperado.

### La diferencia entre el modelo físico y el gemelo digital

- **Modelo físico:** es la explicación matemática del comportamiento normal. Por ejemplo, si el motor recibe más carga, genera más calor; si el ventilador funciona correctamente, parte de ese calor se disipa.
- **Gemelo digital:** es el sistema completo que usa el modelo físico junto con mediciones, estimaciones, indicadores, historial y visualizaciones. El modelo físico es una pieza del gemelo; el gemelo es la representación viva que se actualiza con datos.
- **Equipo real:** es la fuente de la condición que queremos conocer. En este notebook lo reemplazamos por una simulación para poder aprender sin conectar sensores.

Una forma sencilla de verlo es: el modelo físico dice *cómo debería comportarse el motor*; los sensores dicen *cómo se está comportando*; el gemelo compara ambas cosas y ayuda a tomar decisiones.

### Flujo del caso

1. Se simula el motor y sus sensores.
2. Se calcula una expectativa mediante un balance térmico sencillo.
3. Se compara la medición contra esa expectativa.
4. Se estima una condición de salud y se buscan comportamientos inusuales.
5. Se muestran los resultados en gráficas y en una representación visual.

### Supuestos clave

- El motor opera con alimentación estable y la carga cambia con el tiempo.
- La temperatura se aproxima con un modelo de primer orden: el calor generado depende de la carga y de las pérdidas; el enfriamiento depende de la diferencia respecto al ambiente.
- La salud es una señal sintética entre 0 y 1. En un caso real se calibraría con históricos, mantenimiento y pruebas de condición.
- La detección de anomalías es demostrativa y no sustituye una validación industrial.

### Cómo leer los bloques

- **Bloque 1:** prepara las herramientas de Python.
- **Bloque 2:** define el tiempo y las características del motor.
- **Bloque 3:** crea datos simulados, incluyendo un deterioro gradual.
- **Bloque 4:** construye el modelo esperado del gemelo y calcula la salud.
- **Bloque 5:** dibuja las tendencias para ver qué cambió.
- **Bloque 6:** crea un tablero resumido del estado actual.
- **Bloque 7:** muestra el motor de forma esquemática en 3D.
- **Bloque 8:** resume los resultados y propone próximos pasos.

In [ ]:
# 1. Configuración e importaciones
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import IsolationForest

SEED = 42
rng = np.random.default_rng(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)

### 2. Parámetros del sistema físico

El parámetro `thermal_capacity` representa la inercia térmica del motor y `cooling_coefficient` su capacidad de disipación. El modelo no pretende ser un motor industrial exacto; sirve como esqueleto para sustituir parámetros por valores calibrados.

In [ ]:
# 2. Parámetros del motor y del experimento
hours = 24
step_minutes = 5
n = int(hours * 60 / step_minutes)
time = pd.date_range('2026-01-01', periods=n, freq=f'{step_minutes}min')
dt = step_minutes / 60  # horas
ambient_temperature = 25.0
rated_power_kw = 75.0
thermal_capacity = 5.5
cooling_coefficient = 0.24
nominal_efficiency = 0.93
degradation_start = int(n * 0.68)

print(f'Puntos simulados: {n:,}')
print(f'Inicio de degradación: {time[degradation_start]}')

## Data

### 3. Simulación de sensores y fallas incipientes

In [ ]:
# 3. Carga, velocidad y condición real simuladas
t = np.arange(n)
load = 0.58 + 0.14 * np.sin(2 * np.pi * t / 180) + 0.08 * np.sin(2 * np.pi * t / 47)
load += rng.normal(0, 0.025, n)
load = np.clip(load, 0.25, 0.95)

health_true = np.ones(n)
health_true[degradation_start:] = np.linspace(1.0, 0.62, n - degradation_start)
efficiency_true = nominal_efficiency * health_true
friction_factor = 1.0 + (1.0 - health_true) * 1.8
speed_rpm = 1780 - 80 * load - 18 * (1 - health_true) + rng.normal(0, 3.5, n)
power_input_kw = rated_power_kw * load / np.maximum(efficiency_true, 0.55)
vibration_mm_s = 1.15 + 2.6 * (1 - health_true) + 0.75 * load + rng.normal(0, 0.12, n)
vibration_mm_s = np.clip(vibration_mm_s, 0.3, None)

# Balance térmico: temperatura del motor
temperature_true = np.zeros(n)
temperature_true[0] = ambient_temperature + 18
for i in range(1, n):
    heat_generated = 0.34 * power_input_kw[i] * friction_factor[i]
    heat_removed = cooling_coefficient * (temperature_true[i-1] - ambient_temperature)
    temperature_true[i] = temperature_true[i-1] + dt * (heat_generated - heat_removed) / thermal_capacity

sensor_temperature = temperature_true + rng.normal(0, 0.55, n)
sensor_current_a = power_input_kw * 1000 / (np.sqrt(3) * 400 * 0.92) + rng.normal(0, 1.2, n)

sensor_data = pd.DataFrame({
    'timestamp': time,
    'load_fraction': load,
    'speed_rpm': speed_rpm,
    'current_a': sensor_current_a,
    'temperature_c': sensor_temperature,
    'vibration_mm_s': vibration_mm_s,
    'health_true': health_true,
    'efficiency_true': efficiency_true,
})
sensor_data.head()

## Digital twin

### 4. Modelo físico y estimación de salud

El twin recibe carga y temperatura ambiente como entradas. Para cada instante estima una temperatura esperada bajo condición nominal y calcula un índice de salud a partir de la desviación térmica, vibración y eficiencia aparente.

In [ ]:
# 4. Estado esperado por el gemelo digital
twin_temperature = np.zeros(n)
twin_temperature[0] = ambient_temperature + 18
for i in range(1, n):
    nominal_power = rated_power_kw * load[i] / nominal_efficiency
    heat_generated_nominal = 0.34 * nominal_power
    heat_removed = cooling_coefficient * (twin_temperature[i-1] - ambient_temperature)
    twin_temperature[i] = twin_temperature[i-1] + dt * (heat_generated_nominal - heat_removed) / thermal_capacity

thermal_residual = sensor_temperature - twin_temperature
normalized_vibration = np.clip((vibration_mm_s - 1.0) / 4.0, 0, 1)
normalized_temperature = np.clip((thermal_residual + 2) / 20, 0, 1)
health_estimate = 1 - (0.62 * normalized_vibration + 0.38 * normalized_temperature)
health_estimate = pd.Series(health_estimate).rolling(9, min_periods=1).mean().to_numpy()

features = pd.DataFrame({
    'thermal_residual': thermal_residual,
    'vibration_mm_s': vibration_mm_s,
    'current_a': sensor_current_a,
})
anomaly_model = IsolationForest(contamination=0.08, random_state=SEED)
anomaly_model.fit(features.iloc[:degradation_start])
anomaly_score = -anomaly_model.decision_function(features)
anomaly_flag = anomaly_model.predict(features) == -1

twin_data = sensor_data.assign(
    twin_temperature_c=twin_temperature,
    thermal_residual_c=thermal_residual,
    health_estimate=health_estimate,
    anomaly_score=anomaly_score,
    anomaly_flag=anomaly_flag,
)
twin_data.tail()

## Cómo funciona el código, explicado sin tecnicismos

### Bloque 1: herramientas

`numpy` ayuda a trabajar con números y señales; `pandas` organiza los datos como una tabla; `matplotlib` crea gráficas; `plotly` crea gráficas interactivas; `IsolationForest` busca combinaciones poco comunes. No es necesario memorizar estas librerías: cada una cumple una función concreta en el flujo.

### Bloque 2: tiempo y parámetros

Aquí se decide que veremos 24 horas y que habrá una medición cada 5 minutos. También se colocan valores de referencia, como potencia nominal, temperatura ambiente y capacidad térmica. Estos valores son didácticos. Para una aplicación real deberían sustituirse por datos de placa, pruebas y mediciones históricas.

### Bloque 3: datos simulados

La carga sube y baja para imitar un proceso real. Después se agrega ruido, porque ningún sensor mide perfectamente. En el punto marcado como degradación, la salud baja poco a poco. Eso provoca más fricción, menor eficiencia, mayor corriente, más vibración y más temperatura. Así podemos observar si el twin detecta el cambio.

### Bloque 4: el modelo térmico

El código usa una idea intuitiva: la temperatura nueva es igual a la temperatura anterior más el calor que entra menos el calor que sale. La carga y las pérdidas producen calor; el sistema de enfriamiento elimina una parte. El resultado se llama `twin_temperature`. Es la temperatura que el modelo espera cuando el motor está sano.

Luego se calcula `thermal_residual`, que es la diferencia entre la temperatura medida y la temperatura esperada. Un residual pequeño indica que el equipo se parece al comportamiento normal. Un residual creciente indica que hay algo que investigar.

### Bloque 5: salud y anomalías

La salud estimada mezcla dos señales: vibración y diferencia térmica. En esta demostración la vibración pesa un poco más porque suele ser útil para identificar problemas mecánicos. Después `IsolationForest` aprende cómo se ven los datos iniciales, considerados normales, y marca combinaciones raras. Una marca no prueba que exista una falla: significa que conviene revisar ese instante.

### Bloques 6 y 7: visualización

Las gráficas permiten comparar el sensor con el twin. La vista interactiva permite explorar las últimas horas y ver el estado final. El dibujo 3D no pretende ser un diseño CAD; funciona como una pantalla sencilla para comunicar el estado del motor a una persona que no quiere leer todas las tablas.

### Qué cambiaría con un motor real

Se reemplazaría la celda de simulación por una lectura de sensores. El resto del flujo podría mantenerse: limpiar datos, calcular la expectativa, comparar, estimar salud, mostrar el estado y registrar las alertas. También sería necesario validar cada umbral con expertos de mantenimiento.

## Results

### 5. Vista operacional del gemelo digital

In [ ]:
# 5. Tendencias principales
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

axes[0].plot(time, sensor_temperature, label='Sensor', color='#2563eb', linewidth=1.5)
axes[0].plot(time, twin_temperature, label='Gemelo digital', color='#f97316', linewidth=2)
axes[0].axvline(time[degradation_start], color='#dc2626', linestyle='--', label='Inicio degradación')
axes[0].set_ylabel('Temperatura (°C)')
axes[0].set_title('Temperatura medida vs. temperatura esperada')
axes[0].legend(loc='upper left')

axes[1].plot(time, vibration_mm_s, color='#7c3aed', label='Vibración')
axes[1].axhline(3.5, color='#dc2626', linestyle='--', label='Umbral orientativo')
axes[1].set_ylabel('Vibración (mm/s)')
axes[1].set_title('Condición mecánica')
axes[1].legend(loc='upper left')

axes[2].plot(time, health_estimate * 100, color='#059669', linewidth=2, label='Salud estimada')
axes[2].fill_between(time, 0, health_estimate * 100, color='#10b981', alpha=0.15)
axes[2].set_ylabel('Salud (%)')
axes[2].set_ylim(0, 105)
axes[2].set_title('Índice de salud del gemelo digital')
axes[2].legend(loc='upper left')
axes[2].set_xlabel('Tiempo')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Representación visual interactiva del motor
last = twin_data.iloc[-1]
health_pct = float(last['health_estimate'] * 100)
status = 'NORMAL' if health_pct >= 80 else ('VIGILAR' if health_pct >= 65 else 'ALERTA')
status_color = '#16a34a' if status == 'NORMAL' else ('#f59e0b' if status == 'VIGILAR' else '#dc2626')

fig = make_subplots(rows=2, cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}],
           [{'type': 'xy', 'colspan': 2}, None]],
    subplot_titles=('Salud del motor', 'Temperatura actual', 'Últimas 6 horas: sensor vs. gemelo'))

fig.add_trace(go.Indicator(mode='gauge+number', value=health_pct, number={'suffix': '%'},
    title={'text': status}, gauge={'axis': {'range': [0, 100]},
    'bar': {'color': status_color}, 'steps': [
        {'range': [0, 65], 'color': '#fee2e2'},
        {'range': [65, 80], 'color': '#fef3c7'},
        {'range': [80, 100], 'color': '#dcfce7'}]}), row=1, col=1)
fig.add_trace(go.Indicator(mode='number+delta', value=float(last['temperature_c']),
    number={'suffix': ' °C'}, delta={'reference': float(last['twin_temperature_c']), 'relative': False},
    title={'text': 'Medida (delta vs. twin)'}), row=1, col=2)

recent = twin_data.tail(int(6 * 60 / step_minutes))
fig.add_trace(go.Scatter(x=recent['timestamp'], y=recent['temperature_c'], name='Sensor',
    line={'color': '#2563eb', 'width': 2}), row=2, col=1)
fig.add_trace(go.Scatter(x=recent['timestamp'], y=recent['twin_temperature_c'], name='Gemelo digital',
    line={'color': '#f97316', 'width': 2, 'dash': 'dash'}), row=2, col=1)

fig.update_yaxes(title_text='°C', row=2, col=1)
fig.update_layout(height=720, width=1100, title=f'Digital Twin | Estado: {status}',
                  template='plotly_white', legend={'orientation': 'h', 'y': -0.12})
fig.show()

In [ ]:
# 7. Vista 3D mejorada del motor y su estado
# La carcasa es un cilindro, el rotor es otro cilindro interior y las
# luces laterales representan sensores de temperatura y vibración.
theta = np.linspace(0, 2 * np.pi, 80)
z_cylinder = np.linspace(-1.0, 1.0, 30)
theta_grid, z_grid = np.meshgrid(theta, z_cylinder)

def cylinder_surface(radius, z_min, z_max, color, name, opacity=1.0):
    x_surface = radius * np.cos(theta_grid)
    y_surface = radius * np.sin(theta_grid)
    return go.Surface(x=x_surface, y=y_surface, z=z_grid * (z_max - z_min) / 2,
                      surfacecolor=np.zeros_like(x_surface), colorscale=[[0, color], [1, color]],
                      showscale=False, opacity=opacity, name=name, hoverinfo='name')

fig = go.Figure()
fig.add_trace(cylinder_surface(1.25, -1.0, 1.0, '#94a3b8', 'Carcasa', 0.55))
fig.add_trace(cylinder_surface(0.62, -0.9, 0.9, status_color, 'Rotor: condición actual', 0.95))
fig.add_trace(go.Scatter3d(x=[0, 0], y=[0, 0], z=[-1.45, 1.45], mode='lines',
    line={'color': '#334155', 'width': 14}, name='Eje'))

# Aletas sencillas para sugerir enfriamiento
for angle in np.linspace(0, 2 * np.pi, 8, endpoint=False):
    fig.add_trace(go.Scatter3d(x=[1.05*np.cos(angle), 1.45*np.cos(angle)],
        y=[1.05*np.sin(angle), 1.45*np.sin(angle)], z=[0, 0], mode='lines',
        line={'color': '#64748b', 'width': 5}, showlegend=False, hoverinfo='skip'))

sensor_positions = {'Temperatura': (1.42, 0, 0.55), 'Vibración': (0, 1.42, -0.35), 'Corriente': (-1.42, 0, 0.15)}
sensor_colors = {'Temperatura': '#ef4444', 'Vibración': '#8b5cf6', 'Corriente': '#0ea5e9'}
for sensor_name, (sx, sy, sz) in sensor_positions.items():
    fig.add_trace(go.Scatter3d(x=[sx], y=[sy], z=[sz], mode='markers+text',
        marker={'size': 12, 'color': sensor_colors[sensor_name]}, text=[sensor_name],
        textposition='top center', name=f'Sensor: {sensor_name}'))

fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[1.75], mode='text',
    text=[f'<b>{status}</b><br>Salud: {health_pct:.1f}%<br>Temp: {last["temperature_c"]:.1f} °C'],
    textfont={'size': 16, 'color': status_color}, name='Resumen del twin'))
fig.update_layout(title='Gemelo digital del motor: sensores, carcasa, rotor y estado', height=720, width=1050,
    scene={'xaxis': {'title': 'Ancho', 'showbackground': True, 'backgroundcolor': '#f8fafc'},
           'yaxis': {'title': 'Profundidad', 'showbackground': True, 'backgroundcolor': '#f8fafc'},
           'zaxis': {'title': 'Eje vertical', 'showbackground': True, 'backgroundcolor': '#f8fafc'},
           'aspectmode': 'data'}, template='plotly_white', legend={'orientation': 'h'})
fig.show()

### 6. Indicadores finales y takeaways

La siguiente celda resume los valores del último instante simulado y cuenta las observaciones marcadas como anómalas. En una implementación real estos indicadores podrían alimentar alertas, órdenes de mantenimiento o un sistema SCADA/MES.

In [ ]:
summary = pd.Series({
    'Estado final': status,
    'Salud estimada (%)': round(health_pct, 1),
    'Temperatura medida (°C)': round(float(last['temperature_c']), 1),
    'Temperatura del twin (°C)': round(float(last['twin_temperature_c']), 1),
    'Vibración (mm/s)': round(float(last['vibration_mm_s']), 2),
    'Carga (%)': round(float(last['load_fraction']) * 100, 1),
    'Muestras anómalas': int(twin_data['anomaly_flag'].sum()),
})
display(summary.to_frame('Valor'))

print('Conclusiones:')
print(f'- La salud estimada final es {health_pct:.1f}%, con estado {status}.')
print(f"- El residual térmico final es {last['thermal_residual_c']:.1f} °C; aumenta cuando el comportamiento se separa del modelo nominal.")
print(f"- Se marcaron {int(twin_data['anomaly_flag'].sum())} observaciones como potencialmente anómalas.")
print('- Para producción: calibrar el modelo con datos históricos, validar umbrales y conectar sensores en tiempo real.')

## Interpretación sencilla de los resultados

- Al inicio, la línea del sensor y la línea del gemelo deben estar relativamente cerca. Eso significa que el modelo representa razonablemente un motor sano.
- Después de la línea de degradación, la temperatura medida tiende a quedar por encima de la temperatura esperada. Esa separación es el residual térmico: una señal fácil de explicar a mantenimiento.
- La vibración aumenta gradualmente porque la simulación representa fricción o desgaste. En un motor real podría relacionarse con desbalance, desalineación, rodamientos o problemas de montaje, pero el diagnóstico exacto requeriría más evidencia.
- La salud estimada baja porque combina vibración elevada y diferencia térmica. El porcentaje no debe interpretarse como una medición absoluta de vida útil; es un indicador relativo para priorizar revisiones.
- Las marcas de anomalía señalan momentos que se parecen poco al comportamiento aprendido como normal. Una marca debe generar una revisión, no una orden automática de cambiar el motor.

## Conclusiones

1. Un digital twin puede comenzar con un modelo sencillo y datos simulados; no es necesario empezar con una plataforma industrial compleja.
2. La separación entre medición y expectativa es una explicación clara de por qué el sistema considera que algo cambió.
3. El valor principal no está en la imagen 3D aislada, sino en conectar la imagen con sensores, reglas, historial y decisiones.
4. La IA puede ayudar a encontrar patrones raros, pero necesita datos confiables y validación de especialistas.
5. Este notebook es una demostración educativa. Antes de usarlo para seguridad o mantenimiento crítico habría que calibrarlo, probarlo con históricos y definir responsabilidades de operación.

## Próximos pasos

1. Reemplazar la simulación por un CSV o una fuente MQTT/OPC-UA.
2. Ajustar los parámetros térmicos con datos reales de placa y pruebas de operación.
3. Agregar variables como presión, torque, voltaje y consumo energético.
4. Conectar la vista 3D a un selector de tiempo para revisar la evolución histórica.
5. Publicar el dashboard en una aplicación persistente cuando se requiera operación continua.
6. Validar el índice de salud contra mantenimientos y fallas históricas antes de tomar decisiones operativas.